In [1]:
import os
import math
import pandas as pd
import numpy as np
import torch  
from tqdm import tqdm
from torch.utils.data import Dataset, DataLoader  
from tensorflow.keras.preprocessing import text, sequence
from tensorflow.keras.preprocessing.text import Tokenizer
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset, DataLoader  

import time

import pickle
import json

2024-12-19 21:57:33.798739: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2024-12-19 21:57:34.518228: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT


In [2]:
from autoencoder_1 import condition_process,Unet_1d

In [3]:
import copy
from math import sqrt
from random import random
from functools import partial
from contextlib import contextmanager, nullcontext
from typing import List, Union
from collections import namedtuple
from tqdm.auto import tqdm
from functools import partial, wraps

import torch
import torch.nn.functional as F
from torch import nn
from torch.cuda.amp import autocast
from torch.nn.parallel import DistributedDataParallel
import torchvision.transforms as T

import kornia.augmentation as K

from einops import rearrange, repeat, reduce

# helper functions
#写一些辅助函数可以装饰后续函数

#判断变量是否是None，True是非空变量
def exists(val):
    return val is not None

def identity(t, *args, **kwargs):
    return t

#输出数组第一元素，若数组长度为0则输出None
def first(arr, d = None):
    if len(arr) == 0:
        return d
    return arr[0]

#wraps是functiontool函数
#可以保证装饰器修饰的函数的__name__的值和__doc__保持不变
#装饰函数，非空变量会输出代入函数值，空变量则直接返回
def maybe(fn):
    @wraps(fn)
    def inner(x):
        if not exists(x):
            return x
        return fn(x)
    return inner

#装饰函数，调用一次函数，未调用则调用一次，调用过则返回空值
def once(fn):
    called = False
    @wraps(fn)
    def inner(x):
        #nonlocal声明外部嵌套函数内的变量
        nonlocal called
        if called:
            return
        called = True
        return fn(x)
    return inner

#打印一次测试
print_once = once(print)

#pass
#exist判断变量是否是None，True是非空变量
#callable() 函数用于检查一个对象是否是可调用的
def default(val, d):
    if exists(val):
        return val
    return d() if callable(d) else d

#给list变量套个tuple
def cast_tuple(val, length = None):
    #isinstance(object, classinfo) 函数来判断一个对象是否是一个已知的类型，类似 type()
    #isinstance() 会认为子类是一种父类类型，考虑继承关系。
    if isinstance(val, list):
        val = tuple(val)

    output = val if isinstance(val, tuple) else ((val,) * default(length, 1))

    if exists(length):
        assert len(output) == length

    return output

#判断list内元素type是否是float
def is_float_dtype(dtype):
    return any([dtype == float_dtype for float_dtype in (torch.float64, torch.float32, torch.float16, torch.bfloat16)])

#8位图转rgb255图
def cast_uint8_images_to_float(images):
    if not images.dtype == torch.uint8:
        return images
    return images / 255

#tensor所在位置gpu/cpu
def module_device(module):
    return next(module.parameters()).device

#模型参数初始化0
def zero_init_(m):
    nn.init.zeros_(m.weight)
    if exists(m.bias):
        nn.init.zeros_(m.bias)

#装饰函数，用于评估模型
def eval_decorator(fn):
    def inner(model, *args, **kwargs):
        was_training = model.training
        model.eval()
        out = fn(model, *args, **kwargs)
        model.train(was_training)
        return out
    return inner

#把tuple t填满到length长度，填充空字符
def pad_tuple_to_length(t, length, fillvalue = None):
    remain_length = length - len(t)
    if remain_length <= 0:
        return t
    return (*t, *((fillvalue,) * remain_length))

# helper classes
#定义辅助类
class Identity(nn.Module):
    def __init__(self, *args, **kwargs):
        super().__init__()

    def forward(self, x, *args, **kwargs):
        return x

# tensor helpers
#对张量处理的辅助函数

#tensor取自然对数
#clamp（）函数的功能将输入input张量每个元素的值压缩到区间 [min,max]，并返回结果到一个新张量
def log(t, eps: float = 1e-12):
    return torch.log(t.clamp(min = eps))

#L2范数对行归一化，默认2范数
#维度记忆技巧:最后一个维度始终是行，从后向前推:行、列、通道
def l2norm(t):
    return F.normalize(t, dim = -1)

#比较左右张量，给右边张量打扩维补丁？
def right_pad_dims_to(x, t):
    #ndim返回的是数组的维度，返回的只有一个数，该数即表示数组的维度。
    padding_dims = x.ndim - t.ndim
    if padding_dims <= 0:
        return t
    #该函数返回一个有__相同数据__但不同尺寸的 Tensor。通俗一点，就是__改变矩阵维度
    #在形参前加'*'表示可以接受多个实参值存进数组
    #列表或元组前面加星号作用是将列表解开成独立的参数，传入函数；
    #比如原t尺寸（3，3），升4维以后（3，3，1，1），相当于尺寸后面加升维数的1
    #*代表解包实参
    return t.view(*t.shape, *((1,) * padding_dims))

#掩膜
#Python 3 中可以定义一种只能以关键字形式来指定的参数，从而确保调用该函数的代码读起来会比较明确。
#这些参数必须以关键字的形式提供，而不能按位置提供。
#参数列表里的 * 星号，标志着位置参数的就此终结，之后的那些参数，都只能以关键字形式来指定
def masked_mean(t, *, dim, mask = None):
    if not exists(mask):
        return t.mean(dim = dim)

    denom = mask.sum(dim = dim, keepdim = True)
    #升维(b,n) to (b,n,1)
    mask = rearrange(mask, 'b n -> b n 1')
    masked_t = t.masked_fill(~mask, 0.)

    return masked_t.sum(dim = dim) / denom.clamp(min = 1e-5)


#resize原图尺寸到新尺寸
def resize_image_to(
    image,
    target_image_size,
    clamp_range = None
):
    orig_image_size = image.shape[-1]

    if orig_image_size == target_image_size:
        return image
    #利用插值方法，对输入的张量数组进行上\下采样操作，换句话说就是科学合理地改变数组的尺寸大小，尽量保持数据完整
    out = F.interpolate(image.float(), target_image_size, mode = 'linear', align_corners = True) 

    return out


# image normalization functions
# ddpms expect images to be in the range of -1 to 1
def normalize_neg_one_to_one(img):
    return img * 2 - 1

#图片张量反归一化
def unnormalize_zero_to_one(normed_img):
    return (normed_img + 1) * 0.5

# classifier free guidance functions

def prob_mask_like(shape, prob, device):
    if prob == 1:
        return torch.ones(shape, device = device, dtype = torch.bool)
    elif prob == 0:
        return torch.zeros(shape, device = device, dtype = torch.bool)
    else:
        return torch.zeros(shape, device = device).float().uniform_(0, 1) < prob

# gaussian diffusion with continuous time helper functions and classes
# large part of this was thanks to @crowsonkb at https://github.com/crowsonkb/v-diffusion-jax/blob/master/diffusion/utils.py

#torch.jit.script脚本化一个函数或者nn.Module对象，将会检查它的源代码， 将其作为TorchScript代码使用TorchScrit编译器编译它，返回一个ScriptModule或ScriptFunction
#推测根据扩散时长t计算噪音权重alpha和beta,双模式线性或者cos
#snr信噪比？代表扩散过程中，噪音和原数据的权重？
@torch.jit.script
def beta_linear_log_snr(t):#planA
    return -torch.log(torch.special.expm1(1e-4 + 10 * (t ** 2)))#torch.special.expm1指数化-1

@torch.jit.script
def alpha_cosine_log_snr(t, s: float = 0.008):#planB
    return -torch.log((torch.cos((t + s) / (1 + s) * torch.pi * 0.5).pow(-2)) - 1) # not sure if this accounts for beta being clipped to 0.999 in discrete version

#sigmoid激活log_snr
def log_snr_to_alpha_sigma(log_snr):
    return torch.sqrt(torch.sigmoid(log_snr)), torch.sqrt(torch.sigmoid(-log_snr))

class GaussianDiffusionContinuousTimes(nn.Module):
    def __init__(self, *, noise_schedule, timesteps = 1000):
        super().__init__()
        #双模式加噪音

        if noise_schedule == "linear":
            #都是函数，在上面定义的
            self.log_snr = beta_linear_log_snr
        elif noise_schedule == "cosine":
            self.log_snr = alpha_cosine_log_snr
        else:
            raise ValueError(f'invalid noise schedule {noise_schedule}')
        #扩散时间默认1000
        self.num_timesteps = timesteps
    
    #创建一个batch_size尺寸张量，填充noise_level
    def get_times(self, batch_size, noise_level, *, device):
        return torch.full((batch_size,), noise_level, device = device, dtype = torch.float32)
    #为什么训练过程中每一次引入的是随机时间步长 t
    #训练过程中 loss 会逐渐降低，越到后面 loss 的变化幅度越小。如果时间步长 
    #是递增的，那么必然会使得模型过多的关注较早的时间步长（因为早期 loss 大），而忽略了较晚的时间步长信息
    #创建一个batch_size尺寸张量，从0-0.999均匀分布sample
    def sample_random_times(self, batch_size, max_thres = 0.999, *, device):
        return torch.zeros((batch_size,), device = device).float().uniform_(0, max_thres)
    
    #装饰函数maybe，非空变量会输出代入函数值，空变量则直接返回
    #根据上个函数sample出的times（0-1）算出权重参数？
    def get_condition(self, times):
        return maybe(self.log_snr)(times)

    def get_sampling_timesteps(self, batch, *, device):
        #返回一个1维张量，包含在区间start和end上均匀间隔的step个点。输出张量的长度由steps决定。
        times = torch.linspace(1., 0., self.num_timesteps + 1, device = device)
        #用于对张量的某一个维度进行复制，可用于替换pytorch中的repeat
        #将一维t扩增二维，相当于把times行复制batch次，尺寸（batch， times）
        times = repeat(times, 't -> b t', b = batch)
        #沿着一个新维度对输入张量序列进行连接。 序列中所有的张量都应该为相同形状。
        #浅显说法：把多个2维的张量凑成一个3维的张量；多个3维的凑成一个4维的张量…以此类推，也就是在增加新的维度进行堆叠。
        #shape (2, batch, times)
        #[tensor_1, tensor_2,..]或者(tensor_1, tensor_2,..)，且必须tensor_1.shape == tensor_2.shape
        #0-999，和1-1000
        times = torch.stack((times[:, :-1], times[:, 1:]), dim = 0)
        #最后维度（行）上进行拆分，切片，不改变原来的tensor的shape，只是返回展开后的切片
        times = times.unbind(dim = -1)
        return times
    
    #类似编码器先验q(zt|x)?
    #之前定义的default辅助函数
    #xstart原图，xt噪音图
    #若没有安排tnext则按函数lambda计算tnext
    def q_posterior(self, x_start, x_t, t, *, t_next = None):
        t_next = default(t_next, lambda: (t - 1. / self.num_timesteps).clamp(min = 0.))
        # clamp（）函数的功能将输入input张量每个元素的值压缩到区间 [min,max],min：限制范围下限；扩散时间默认1000
        """ https://openreview.net/attachment?id=2LdBqxc1Yv&name=supplementary_material """
        log_snr = self.log_snr(t)
        log_snr_next = self.log_snr(t_next)
        #functiontool.patial,固定x_t参数，给函数打补丁补全维度，log_snr, log_snr_next
        #对比x_t，给升维log_snr, log_snr_next
        log_snr, log_snr_next = map(partial(right_pad_dims_to, x_t), (log_snr, log_snr_next))

        alpha, sigma = log_snr_to_alpha_sigma(log_snr)
        alpha_next, sigma_next = log_snr_to_alpha_sigma(log_snr_next)

        # c - as defined near eq 33
        c = -expm1(log_snr - log_snr_next)
        posterior_mean = alpha_next * (x_t * (1 - c) / alpha + c * x_start)

        # following (eq. 33)
        posterior_variance = (sigma_next ** 2) * c
        posterior_log_variance_clipped = log(posterior_variance, eps = 1e-20)
        return posterior_mean, posterior_variance, posterior_log_variance_clipped
    #重参数化技巧，由先验分布sample一个
    def q_sample(self, x_start, t, noise = None):
        dtype = x_start.dtype
        #isinstance(object, classinfo) 函数来判断一个对象是否是一个已知的类型，类似 type()

        if isinstance(t, float):
            batch = x_start.shape[0]
            #形状(batch,)，用t填充
            t = torch.full((batch,), t, device = x_start.device, dtype = dtype)
        #返回一个和输入大小相同的张量，其由均值为0、方差为1的标准正态分布填充
        #reparametrize，sample一个和x0一样尺寸的N(0,1)
        noise = default(noise, lambda: torch.randn_like(x_start))
        log_snr = self.log_snr(t).type(dtype)
        log_snr_padded_dim = right_pad_dims_to(x_start, log_snr)
        alpha, sigma =  log_snr_to_alpha_sigma(log_snr_padded_dim)

        return alpha * x_start + sigma * noise, log_snr
    #从xi sample一个到xj
    def q_sample_from_to(self, x_from, from_t, to_t, noise = None):
        shape, device, dtype = x_from.shape, x_from.device, x_from.dtype
        batch = shape[0]

        if isinstance(from_t, float):
            from_t = torch.full((batch,), from_t, device = device, dtype = dtype)

        if isinstance(to_t, float):
            to_t = torch.full((batch,), to_t, device = device, dtype = dtype)

        noise = default(noise, lambda: torch.randn_like(x_from))

        log_snr = self.log_snr(from_t)
        log_snr_padded_dim = right_pad_dims_to(x_from, log_snr)
        alpha, sigma =  log_snr_to_alpha_sigma(log_snr_padded_dim)

        log_snr_to = self.log_snr(to_t)
        log_snr_padded_dim_to = right_pad_dims_to(x_from, log_snr_to)
        alpha_to, sigma_to =  log_snr_to_alpha_sigma(log_snr_padded_dim_to)

        return x_from * (alpha_to / alpha) + noise * (sigma_to * alpha - sigma * alpha_to) / alpha
    #去噪
    def predict_start_from_noise(self, x_t, t, noise):
        log_snr = self.log_snr(t)
        log_snr = right_pad_dims_to(x_t, log_snr)
        alpha, sigma = log_snr_to_alpha_sigma(log_snr)
        return (x_t - sigma * noise) / alpha.clamp(min = 1e-8)

/work/home/af2/miniconda3/envs/cont_diffusion/lib/python3.8/site-packages/torchvision/io/image.py:13: UserWarning: Failed to load image Python extension: /work/home/af2/miniconda3/envs/cont_diffusion/lib/python3.8/site-packages/torchvision/image.so: undefined symbol: _ZN2at4_ops19empty_memory_format4callEN3c108ArrayRefIlEENS2_8optionalINS2_10ScalarTypeEEENS5_INS2_6LayoutEEENS5_INS2_6DeviceEEENS5_IbEENS5_INS2_12MemoryFormatEEE
  warn(f"Failed to load image Python extension: {e}")


In [4]:
# null unet

class NullUnet(nn.Module):
    def __init__(self, *args, **kwargs):
        super().__init__()
        self.lowres_cond = False
        self.dummy_parameter = nn.Parameter(torch.tensor([0.]))

    def cast_model_parameters(self, *args, **kwargs):
        return self

    def forward(self, x, *args, **kwargs):
        return x

In [5]:
# constants

Hparams_fields = [
    'num_sample_steps',
    'sigma_min',
    'sigma_max',
    'sigma_data',
    'rho',
    'P_mean',
    'P_std',
    'S_churn',
    'S_tmin',
    'S_tmax',
    'S_noise'
]

Hparams = namedtuple('Hparams', Hparams_fields)

# helper functions

def log(t, eps = 1e-20):
    return torch.log(t.clamp(min = eps))

# main class

class ElucidatedImagen(nn.Module):
    def __init__(
        self,
        unets,
        *,
        image_sizes,                                # for cascading ddpm, image size at each stage
        text_encoder_name = '',
        text_embed_dim =512,
        channels = 3,
        channels_out=3,
        cond_drop_prob = 0.1,
        random_crop_sizes = None,
        lowres_sample_noise_level = 0.2,            # in the paper, they present a new trick where they noise the lowres conditioning image, and at sample time, fix it to a certain level (0.1 or 0.3) - the unets are also made to be conditioned on this noise level
        per_sample_random_aug_noise_level = False,  # unclear when conditioning on augmentation noise level, whether each batch element receives a random aug noise value - turning off due to @marunine's find
        condition_on_text = True,
        auto_normalize_img = True,                  # whether to take care of normalizing the image from [0, 1] to [-1, 1] and back automatically - you can turn this off if you want to pass in the [-1, 1] ranged image yourself from the dataloader
        dynamic_thresholding = True,
        dynamic_thresholding_percentile = 0.95,     # unsure what this was based on perusal of paper
        only_train_unet_number = None,
        lowres_noise_schedule = 'linear',
        num_sample_steps = 32,                      # number of sampling steps
        sigma_min = 0.002,                          # min noise level
        sigma_max = 80,                             # max noise level
        sigma_data = 0.5,                           # standard deviation of data distribution
        rho = 7,                                    # controls the sampling schedule
        P_mean = -1.2,                              # mean of log-normal distribution from which noise is drawn for training
        P_std = 1.2,                                # standard deviation of log-normal distribution from which noise is drawn for training
        S_churn = 80,                               # parameters for stochastic sampling - depends on dataset, Table 5 in apper
        S_tmin = 0.05,
        S_tmax = 50,
        S_noise = 1.003,
        #categorical_loss = False,
        loss_type=0, #0=MSE, 1=Cross entropy, 2=KLDIv Loss
        categorical_loss_ignore=None,
    ):
        super().__init__()

        self.only_train_unet_number = only_train_unet_number

        # conditioning hparams

        self.condition_on_text = condition_on_text
        self.unconditional = not condition_on_text
        self.loss_type=loss_type
        if self.loss_type>0:
            self.categorical_loss=True
            self.m = nn.LogSoftmax(dim=1) #used for some loss functins
        else:
            self.categorical_loss=False
            
        print ("Loss type: ", self.loss_type)
        self.categorical_loss_ignore=categorical_loss_ignore
        
        # channels

        self.channels = channels
        self.channels_out = channels_out
        

        # automatically take care of ensuring that first unet is unconditional
        # while the rest of the unets are conditioned on the low resolution image produced by previous unet

        unets = cast_tuple(unets)
        num_unets = len(unets)

        # randomly cropping for upsampler training

        self.random_crop_sizes = cast_tuple(random_crop_sizes, num_unets)
        assert not exists(first(self.random_crop_sizes)), 'you should not need to randomly crop image during training for base unet, only for upsamplers - so pass in `random_crop_sizes = (None, 128, 256)` as example'

        # lowres augmentation noise schedule

        self.lowres_noise_schedule = GaussianDiffusionContinuousTimes(noise_schedule = lowres_noise_schedule)
      
        self.text_embed_dim = text_embed_dim
        
        # construct unets

        self.unets = nn.ModuleList([])
        self.unet_being_trained_index = -1 # keeps track of which unet is being trained at the moment

        print (f"Channels in={self.channels}, channels out={self.channels_out}")
        for ind, one_unet in enumerate(unets):
            
            assert isinstance(one_unet, (Unet_1d, NullUnet))
            is_first = ind == 0

            one_unet = one_unet.cast_model_parameters(
                lowres_cond = not is_first,
                cond_on_text = self.condition_on_text,
                text_embed_dim = self.text_embed_dim if self.condition_on_text else None,
                channels = self.channels,
                #channels_out = self.channels
                channels_out = self.channels_out
            )

            self.unets.append(one_unet)

        # determine whether we are training on images or video

        is_video = False  
        self.is_video = is_video
        # self.right_pad_dims_to_datatype = partial(rearrange, pattern = ('b -> b 1 1' if not is_video else 'b -> b 1 1 1'))
        self.right_pad_dims_to_datatype = partial(rearrange, pattern = ('b -> b 1' if not is_video else 'b -> b 1 1 1'))
        self.resize_to = resize_video_to if is_video else resize_image_to

        # unet image sizes

         
        self.image_sizes = image_sizes
        assert num_unets == len(self.image_sizes), f'you did not supply the correct number of u-nets ({len(self.unets)}) for resolutions {self.image_sizes}'

        self.sample_channels = cast_tuple(self.channels, num_unets)

        # cascading ddpm related stuff

        lowres_conditions = tuple(map(lambda t: t.lowres_cond, self.unets))
        assert lowres_conditions == (False, *((True,) * (num_unets - 1))), 'the first unet must be unconditioned (by low resolution image), and the rest of the unets must have `lowres_cond` set to True'

        self.lowres_sample_noise_level = lowres_sample_noise_level
        self.per_sample_random_aug_noise_level = per_sample_random_aug_noise_level

        # classifier free guidance

        self.cond_drop_prob = cond_drop_prob
        self.can_classifier_guidance = cond_drop_prob > 0.

        # normalize and unnormalize image functions

        self.normalize_img = normalize_neg_one_to_one if auto_normalize_img else identity
        self.unnormalize_img = unnormalize_zero_to_one if auto_normalize_img else identity
        self.input_image_range = (0. if auto_normalize_img else -1., 1.)

        # dynamic thresholding

        self.dynamic_thresholding = cast_tuple(dynamic_thresholding, num_unets)
        self.dynamic_thresholding_percentile = dynamic_thresholding_percentile

        # elucidating parameters

        hparams = [
            num_sample_steps,
            sigma_min,
            sigma_max,
            sigma_data,
            rho,
            P_mean,
            P_std,
            S_churn,
            S_tmin,
            S_tmax,
            S_noise,
        ]

        hparams = [cast_tuple(hp, num_unets) for hp in hparams]
        self.hparams = [Hparams(*unet_hp) for unet_hp in zip(*hparams)]

        # one temp parameter for keeping track of device

        self.register_buffer('_temp', torch.tensor([0.]), persistent = False)

        # default to device of unets passed in

        self.to(next(self.unets.parameters()).device)

    def force_unconditional_(self):
        self.condition_on_text = False
        self.unconditional = True

        for unet in self.unets:
            unet.cond_on_text = False

    @property
    def device(self):
        return self._temp.device

    def get_unet(self, unet_number):
        assert 0 < unet_number <= len(self.unets)
        index = unet_number - 1

        if isinstance(self.unets, nn.ModuleList):
            unets_list = [unet for unet in self.unets]
            delattr(self, 'unets')
            self.unets = unets_list

        if index != self.unet_being_trained_index:
            for unet_index, unet in enumerate(self.unets):
                unet.to(self.device if unet_index == index else 'cpu')

        self.unet_being_trained_index = index
        return self.unets[index]

    def reset_unets_all_one_device(self, device = None):
        device = default(device, self.device)
        self.unets = nn.ModuleList([*self.unets])
        self.unets.to(device)

        self.unet_being_trained_index = -1

    @contextmanager
    def one_unet_in_gpu(self, unet_number = None, unet = None):
        assert exists(unet_number) ^ exists(unet)

        if exists(unet_number):
            unet = self.unets[unet_number - 1]

        devices = [module_device(unet) for unet in self.unets]
        self.unets.cpu()
        unet.to(self.device)

        yield

        for unet, device in zip(self.unets, devices):
            unet.to(device)

    # overriding state dict functions

    def state_dict(self, *args, **kwargs):
        self.reset_unets_all_one_device()
        return super().state_dict(*args, **kwargs)

    def load_state_dict(self, *args, **kwargs):
        self.reset_unets_all_one_device()
        return super().load_state_dict(*args, **kwargs)

    # dynamic thresholding

    def threshold_x_start(self, x_start, dynamic_threshold = True):
        if not dynamic_threshold:
            # return x_start.clamp(0., 1.)
            return x_start.clamp(-1, 1.)
            # return (x_start + 1.) / 2.  

        s = torch.quantile(
            rearrange(x_start, 'b ... -> b (...)').abs(),
            self.dynamic_thresholding_percentile,
            dim = -1
        )

        s.clamp_(min = 1.)
        s = right_pad_dims_to(x_start, s)
        return x_start.clamp(-s, s) / s
        # return x_start.clamp(0., s) / s
        # return (x_start.clamp(-s, s) / s + 1.) / 2.

    # derived preconditioning params - Table 1

    def c_skip(self, sigma_data, sigma):
        return (sigma_data ** 2) / (sigma ** 2 + sigma_data ** 2)

    def c_out(self, sigma_data, sigma):
        return sigma * sigma_data * (sigma_data ** 2 + sigma ** 2) ** -0.5

    def c_in(self, sigma_data, sigma):
        return 1 * (sigma ** 2 + sigma_data ** 2) ** -0.5

    def c_noise(self, sigma):
        return log(sigma) * 0.25
        # return sigma

   # preconditioned network output
    # equation (7) in the paper

    def preconditioned_network_forward(
        self,
        unet_forward,
        noised_images,
        sigma,
        *,
        sigma_data,
        clamp = False,
        dynamic_threshold = True,
        **kwargs
    ):
        batch, device = noised_images.shape[0], noised_images.device

        if isinstance(sigma, float):
            sigma = torch.full((1,), sigma, device = device)
        # print('pnf:', sigma.shape)
        padded_sigma = self.right_pad_dims_to_datatype(sigma)

        net_out = unet_forward(
            self.c_in(sigma_data, padded_sigma) * noised_images,
            self.c_noise(sigma),
            **kwargs
        )

        out = self.c_skip(sigma_data, padded_sigma) * noised_images +  self.c_out(sigma_data, padded_sigma) * net_out

        if not clamp:
            return out

        return self.threshold_x_start(out, dynamic_threshold)

    # sampling

    # sample schedule
    # equation (5) in the paper

    def sample_schedule(
        self,
        num_sample_steps,
        rho,
        sigma_min,
        sigma_max
    ):
        N = num_sample_steps
        inv_rho = 1 / rho

        steps = torch.arange(num_sample_steps, device = self.device, dtype = torch.float32)
        sigmas = (sigma_max ** inv_rho + steps / (N - 1) * (sigma_min ** inv_rho - sigma_max ** inv_rho)) ** rho

        sigmas = F.pad(sigmas, (0, 1), value = 0.) # last step is sigma value of 0.
        return sigmas

    @torch.no_grad()
    def one_unet_sample(
        self,
        unet,
        shape,
        *,
        unet_number,
        clamp = True,
        dynamic_threshold = True,
        cond_scale = 1.,
        use_tqdm = True,
        inpaint_images = None,
        inpaint_masks = None,
        inpaint_resample_times = 5,
        init_images = None,
        skip_steps = None,
        sigma_min = None,
        sigma_max = None,
        **kwargs
    ):
        # get specific sampling hyperparameters for unet

        hp = self.hparams[unet_number - 1]

        sigma_min = default(sigma_min, hp.sigma_min)
        sigma_max = default(sigma_max, hp.sigma_max)

        # get the schedule, which is returned as (sigma, gamma) tuple, and pair up with the next sigma and gamma

        sigmas = self.sample_schedule(hp.num_sample_steps, hp.rho, sigma_min, sigma_max)
        # 输出一个随着steps增加递减序列 (sigma_max, ..., sigma_min, 0)
        # print('sample sigmas:', sigmas)
        # print('sample sigmas:', sigmas.shape)
        gammas = torch.where(
            (sigmas >= hp.S_tmin) & (sigmas <= hp.S_tmax),
            min(hp.S_churn / hp.num_sample_steps, sqrt(2) - 1),
            0.
        )

        sigmas_and_gammas = list(zip(sigmas[:-1], sigmas[1:], gammas[:-1]))

        # images is noise at the beginning

        init_sigma = sigmas[0]

        images = init_sigma * torch.randn(shape, device = self.device)

        # initializing with an image

        if exists(init_images):
            images += init_images

        # keeping track of x0, for self conditioning if needed

        x_start = None

        # prepare inpainting images and mask

        has_inpainting = exists(inpaint_images) and exists(inpaint_masks)
        resample_times = inpaint_resample_times if has_inpainting else 1

        if has_inpainting:
            inpaint_images = self.normalize_img(inpaint_images)
            inpaint_images = self.resize_to(inpaint_images, shape[-1])
            inpaint_masks = self.resize_to(rearrange(inpaint_masks, 'b ... -> b 1 ...').float(), shape[-1]).bool()

        # unet kwargs

        unet_kwargs = dict(
            sigma_data = hp.sigma_data,
            clamp = clamp,
            dynamic_threshold = dynamic_threshold,
            cond_scale = cond_scale,
            **kwargs
        )

        # gradually denoise

        initial_step = default(skip_steps, 0)
        sigmas_and_gammas = sigmas_and_gammas[initial_step:]

        total_steps = len(sigmas_and_gammas)

        for ind, (sigma, sigma_next, gamma) in tqdm(enumerate(sigmas_and_gammas), total = total_steps, desc = 'sampling time step', disable = not use_tqdm):
            is_last_timestep = ind == (total_steps - 1)

            sigma, sigma_next, gamma = map(lambda t: t.item(), (sigma, sigma_next, gamma))

            for r in reversed(range(resample_times)):
                is_last_resample_step = r == 0

                eps = hp.S_noise * torch.randn(shape, device = self.device) # stochastic sampling

                sigma_hat = sigma + gamma * sigma
                added_noise = sqrt(sigma_hat ** 2 - sigma ** 2) * eps

                images_hat = images + added_noise

                self_cond = x_start if unet.self_cond else None

                if has_inpainting:
                    images_hat = images_hat * ~inpaint_masks + (inpaint_images + added_noise) * inpaint_masks

                model_output = self.preconditioned_network_forward(
                    unet.forward_with_cond_scale,
                    images_hat,
                    sigma_hat,
                    **unet_kwargs
                )

                # print('model_output:')
                # print(model_output)
                denoised_over_sigma = (images_hat - model_output) / sigma_hat
                
                images_next = images_hat + (sigma_next - sigma_hat) * denoised_over_sigma

                # second order correction, if not the last timestep
                #图的二阶矫正可选

                if sigma_next != 0:
                    self_cond = model_output if unet.self_cond else None

                    model_output_next = self.preconditioned_network_forward(
                        unet.forward_with_cond_scale,
                        images_next,
                        sigma_next,
                        **unet_kwargs
                    )
                    
                    denoised_prime_over_sigma = (images_next - model_output_next) / sigma_next
                    images_next = images_hat + 0.5 * (sigma_next - sigma_hat) * (denoised_over_sigma + denoised_prime_over_sigma)
                # print('images_next:')
                # print(images_next)
                images = images_next

                if has_inpainting and not (is_last_resample_step or is_last_timestep):
                    # renoise in repaint and then resample
                    repaint_noise = torch.randn(shape, device = self.device)
                    images = images + (sigma - sigma_next) * repaint_noise

                x_start = model_output  # save model output for self conditioning
                

        if has_inpainting:
            images = images * ~inpaint_masks + inpaint_images * inpaint_masks

        return  images

    @torch.no_grad()
    @eval_decorator
    def sample(
        self,
        texts: List[str] = None,
        text_masks = None,
        text_embeds = None,
        cond_images = None,
        inpaint_images = None,
        inpaint_masks = None,
        inpaint_resample_times = 5,
        init_images = None,
        skip_steps = None,
        sigma_min = None,
        sigma_max = None,
        video_frames = None,
        batch_size = 1,
        cond_scale = 1.,
        lowres_sample_noise_level = None,
        start_at_unet_number = 1,
        start_image_or_video = None,
        stop_at_unet_number = None,
        return_all_unet_outputs = False,
        return_pil_images = False,
        use_tqdm = True,
        device = None,
        
    ):
        device = default(device, self.device)
        self.reset_unets_all_one_device(device = device)

        cond_images = maybe(cast_uint8_images_to_float)(cond_images)

        if exists(texts) and not exists(text_embeds) and not self.unconditional:
            assert all([*map(len, texts)]), 'text cannot be empty'

            with autocast(enabled = False):
                text_embeds, text_masks = self.encode_text(texts, return_attn_mask = True)

            text_embeds, text_masks = map(lambda t: t.to(device), (text_embeds, text_masks))

        if not self.unconditional:
            assert exists(text_embeds), 'text must be passed in if the network was not trained without text `condition_on_text` must be set to `False` when training'

            text_masks = default(text_masks, lambda: torch.any(text_embeds != 0., dim = -1))
            batch_size = text_embeds.shape[0]

        if exists(inpaint_images):
            if self.unconditional:
                if batch_size == 1: # assume researcher wants to broadcast along inpainted images
                    batch_size = inpaint_images.shape[0]

            assert inpaint_images.shape[0] == batch_size, 'number of inpainting images must be equal to the specified batch size on sample `sample(batch_size=<int>)``'
            assert not (self.condition_on_text and inpaint_images.shape[0] != text_embeds.shape[0]), 'number of inpainting images must be equal to the number of text to be conditioned on'

        assert not (self.condition_on_text and not exists(text_embeds)), 'text or text encodings must be passed into imagen if specified'
        assert not (not self.condition_on_text and exists(text_embeds)), 'imagen specified not to be conditioned on text, yet it is presented'
        assert not (exists(text_embeds) and text_embeds.shape[-1] != self.text_embed_dim), f'invalid text embedding dimension being passed in (should be {self.text_embed_dim})'

        assert not (exists(inpaint_images) ^ exists(inpaint_masks)),  'inpaint images and masks must be both passed in to do inpainting'

        outputs = []

        is_cuda = next(self.parameters()).is_cuda
        device = next(self.parameters()).device

        lowres_sample_noise_level = default(lowres_sample_noise_level, self.lowres_sample_noise_level)

        num_unets = len(self.unets)
        cond_scale = cast_tuple(cond_scale, num_unets)

        # handle video and frame dimension

        assert not (self.is_video and not exists(video_frames)), 'video_frames must be passed in on sample time if training on video'

        frame_dims = (video_frames,) if self.is_video else tuple()

        # initializing with an image or video

        init_images = cast_tuple(init_images, num_unets)
        init_images = [maybe(self.normalize_img)(init_image) for init_image in init_images]

        skip_steps = cast_tuple(skip_steps, num_unets)

        sigma_min = cast_tuple(sigma_min, num_unets)
        sigma_max = cast_tuple(sigma_max, num_unets)

        # handle starting at a unet greater than 1, for training only-upscaler training

        if start_at_unet_number > 1:
            assert start_at_unet_number <= num_unets, 'must start a unet that is less than the total number of unets'
            assert not exists(stop_at_unet_number) or start_at_unet_number <= stop_at_unet_number
            assert exists(start_image_or_video), 'starting image or video must be supplied if only doing upscaling'

            prev_image_size = self.image_sizes[start_at_unet_number - 2]
            img = self.resize_to(start_image_or_video, prev_image_size)

        
        for unet_number, unet, channel, image_size, unet_hparam, dynamic_threshold, unet_cond_scale, unet_init_images, unet_skip_steps, unet_sigma_min, unet_sigma_max in tqdm(zip(range(1, num_unets + 1), self.unets, self.sample_channels, self.image_sizes, self.hparams, self.dynamic_thresholding, cond_scale, init_images, skip_steps, sigma_min, sigma_max), disable = not use_tqdm):
            if unet_number < start_at_unet_number:
                continue

            assert not isinstance(unet, NullUnet), 'cannot sample from null unet'

            context = self.one_unet_in_gpu(unet = unet) if is_cuda else nullcontext()

            with context:
                lowres_cond_img = lowres_noise_times = None

                shape = (batch_size, channel, *frame_dims, image_size )

                if unet.lowres_cond:
                    lowres_noise_times = self.lowres_noise_schedule.get_times(batch_size, lowres_sample_noise_level, device = device)

                    lowres_cond_img = self.resize_to(img, image_size)
                    
                   
                    lowres_cond_img = self.normalize_img(lowres_cond_img.float())

                    lowres_cond_img, _ = self.lowres_noise_schedule.q_sample(x_start = lowres_cond_img.float(), 
                                                                             t = lowres_noise_times, 
                                                                             noise = torch.randn_like(lowres_cond_img.float()))

                if exists(unet_init_images):
                    unet_init_images = self.resize_to(unet_init_images, image_size)

                #shape = (batch_size, self.channels, *frame_dims, image_size, image_size)
                shape = (batch_size, image_size)

                img = self.one_unet_sample(
                    unet,
                    shape,
                    unet_number = unet_number,
                    text_embeds = text_embeds,
                    text_mask =text_masks,
                    cond_images = cond_images,
                    inpaint_images = inpaint_images,
                    inpaint_masks = inpaint_masks,
                    inpaint_resample_times = inpaint_resample_times,
                    init_images = unet_init_images,
                    skip_steps = unet_skip_steps,
                    sigma_min = unet_sigma_min,
                    sigma_max = unet_sigma_max,
                    cond_scale = unet_cond_scale,
                    lowres_cond_img = lowres_cond_img,
                    lowres_noise_times = lowres_noise_times,
                    dynamic_threshold = dynamic_threshold,
                    use_tqdm = use_tqdm
                )
 
                if self.categorical_loss:
                    img=self.m(img)
                outputs.append(img)

            if exists(stop_at_unet_number) and stop_at_unet_number == unet_number:
                break

        output_index = -1 if not return_all_unet_outputs else slice(None) # either return last unet output or all unet outputs


        if not return_all_unet_outputs:
            outputs = outputs[-1:]

        assert not self.is_video, 'automatically converting video tensor to video file for saving is not built yet'
        
         
        if self.categorical_loss:
            return  torch.argmax(outputs[output_index], dim=1).unsqueeze (1)
        else:
            return  outputs[output_index]
       
    # training

    def loss_weight(self, sigma_data, sigma):
        return (sigma ** 2 + sigma_data ** 2) * (sigma * sigma_data) ** -2

    def noise_distribution(self, P_mean, P_std, batch_size):
        return (P_mean + P_std * torch.randn((batch_size,), device = self.device)).exp()

    def forward(
        self,
        images,
        unet: Union[Unet_1d, NullUnet, DistributedDataParallel] = None,
        texts: List[str] = None,
        text_embeds = None,
        text_masks = None,
        unet_number = None,
        cond_images = None,
       
    ):
        assert not (len(self.unets) > 1 and not exists(unet_number)), f'you must specify which unet you want trained, from a range of 1 to {len(self.unets)}, if you are training cascading DDPM (multiple unets)'
        unet_number = default(unet_number, 1)
        assert not exists(self.only_train_unet_number) or self.only_train_unet_number == unet_number, 'you can only train on unet #{self.only_train_unet_number}'

       
        cond_images = maybe(cast_uint8_images_to_float)(cond_images)
        
        if self.categorical_loss==False:
           pass 

        unet_index = unet_number - 1
        
        unet = default(unet, lambda: self.get_unet(unet_number))

        assert not isinstance(unet, NullUnet), 'null unet cannot and should not be trained'

        target_image_size    = self.image_sizes[unet_index]
        random_crop_size     = self.random_crop_sizes[unet_index]
        prev_image_size      = self.image_sizes[unet_index - 1] if unet_index > 0 else None
        hp                   = self.hparams[unet_index]
        # print('images.shape', images.shape)
        batch_size, seq_len, device, is_video = *images.shape, images.device, (images.ndim == 4)

        frames = images.shape[2] if is_video else None


        if exists(texts) and not exists(text_embeds) and not self.unconditional:
            assert all([*map(len, texts)]), 'text cannot be empty'
            assert len(texts) == len(images), 'number of text captions does not match up with the number of images given'

            with autocast(enabled = False):
                text_embeds, text_masks = self.encode_text(texts, return_attn_mask = True)

            text_embeds, text_masks = map(lambda t: t.to(images.device), (text_embeds, text_masks))

        if not self.unconditional:
            text_masks = default(text_masks, lambda: torch.any(text_embeds != 0., dim = -1))

        assert not (self.condition_on_text and not exists(text_embeds)), 'text or text encodings must be passed into decoder if specified'
        assert not (not self.condition_on_text and exists(text_embeds)), 'decoder specified not to be conditioned on text, yet it is presented'

        assert not (exists(text_embeds) and text_embeds.shape[-1] != self.text_embed_dim), f'invalid text embedding dimension being passed in (should be {self.text_embed_dim})'

        lowres_cond_img = lowres_aug_times = None
        if exists(prev_image_size):
            lowres_cond_img = self.resize_to(images, prev_image_size, clamp_range = self.input_image_range)
            lowres_cond_img = self.resize_to(lowres_cond_img, target_image_size, clamp_range = self.input_image_range)

            if self.per_sample_random_aug_noise_level:
                lowres_aug_times = self.lowres_noise_schedule.sample_random_times(batch_size, device = device)
            else:
                lowres_aug_time = self.lowres_noise_schedule.sample_random_times(1, device = device)
                lowres_aug_times = repeat(lowres_aug_time, '1 -> b', b = batch_size)

        # print('lowres_aug_time', lowres_aug_time)
        # print('lowres_aug_times:', lowres_aug_times)
        if exists(random_crop_size):
            aug = K.RandomCrop((random_crop_size, random_crop_size), p = 1.)

            if is_video:
                images, lowres_cond_img = rearrange_many((images, lowres_cond_img), 'b c f h -> (b f) c h')

            images = aug(images)
            lowres_cond_img = aug(lowres_cond_img, params = aug._params)

            if is_video:
                images, lowres_cond_img = rearrange_many((images, lowres_cond_img), '(b f) c h -> b c f h', f = frames)

        # noise the lowres conditioning image
        # at sample time, they then fix the noise level of 0.1 - 0.3

        lowres_cond_img_noisy = None
        if exists(lowres_cond_img):
            lowres_cond_img_noisy, _ = self.lowres_noise_schedule.q_sample(x_start = lowres_cond_img, 
                                                                           t = lowres_aug_times,
                                                                           noise = torch.randn_like(lowres_cond_img.float()))

        # get the sigmas

        sigmas = self.noise_distribution(hp.P_mean, hp.P_std, 1)
        # sigmas = self.noise_distribution(hp.P_mean, hp.P_std, batch_size)
        padded_sigmas = self.right_pad_dims_to_datatype(sigmas)
        # print('sigma:', sigmas.shape)
        # print('padded_sigmas:', padded_sigmas.shape)
        # noise

        noise = torch.randn_like(images.float())
        # print('noise:', noise.shape)
        noised_images = images + padded_sigmas * noise  # alphas are 1. in the paper
        # print('noised images:', noised_images.shape)
        # print('images:', images.shape)

        # print('elucidetedimagen text_emb:', text_embeds.dtype, text_embeds.shape)
        unet_kwargs = dict(
            sigma_data = hp.sigma_data,
            text_embeds = text_embeds,
            text_mask = text_masks,
            cond_images = cond_images,
            lowres_noise_times = self.lowres_noise_schedule.get_condition(lowres_aug_times),
            lowres_cond_img = lowres_cond_img_noisy,
            cond_drop_prob = self.cond_drop_prob,
        )
        # print(unet_kwargs)
        
        self_cond = unet.module.self_cond if isinstance(unet, DistributedDataParallel) else unet
        # print(sigmas.shape)
        if self_cond and random() < 0.5:
            with torch.no_grad():
                pred_x0 = self.preconditioned_network_forward(
                    unet.forward,
                    noised_images,
                    sigmas,
                    **unet_kwargs
                ).detach()

            unet_kwargs = {**unet_kwargs, 'self_cond': pred_x0}

        # get prediction
        # print('sigmas:', sigmas.shape)
        # print(sigmas)
        denoised_images = self.preconditioned_network_forward(
            unet.forward,
            noised_images,
            sigmas,
            **unet_kwargs
        )
      
        if self.loss_type==0: #self.categorical_loss == False:

            losses = F.mse_loss(denoised_images, images, reduction = 'none')
            losses = reduce(losses, 'b ... -> b', 'mean')

            # loss weighting

            losses = losses * self.loss_weight(hp.sigma_data, sigmas)
            losses = losses.mean()
        
       

        return losses 

In [6]:
class PPDesigner(nn.Module):
    def __init__(self, 
                 text_embed_dim =256,
                 vocal_size=22,
                 timesteps=(64), 
                 max_length=250,
                 pred_dim=1,
                 loss_type=0, 
                 elucidated=True,  
                 device='cuda:2'
                ):
        super().__init__()
        
        self.device = torch.device(device) 
        self.pred_dim = pred_dim
        self.loss_type = loss_type
        self.is_elucidated = True
        self.condition_process = condition_process(
            seq_len=max_length,
            in_features=1, # 32
            hidden_channels=256, # 64
            out_feature=text_embed_dim, # 64
            dropout=0.1
        ).to(self.device)  #output size (batch, seq, out_feature)
        
        unet1 = Unet_1d(
             self_cond=False,
                 seq_len=max_length,
                 condition_feature=text_embed_dim,
                 time_in_feature=1,
                 time_hidden_feature=512,
                 time_out_feature=512,
                 dropout=0.1,
                 fc_h=512,
                 fc_out=1024,
                 num_blocks=5, #transformer的encoder数量5+1=6
                 num_heads=8,
                 d_encoder_ff=2048, #前馈网络隐藏层维度
                 in_hidden_channel=1024, #噪音张量的隐藏层特征数量
                 in_feature=1024, #噪音张量输出特征数量
                 lstm_hidden_size=512,
                 lstm_num_layers=5,
                 batch_first=True,
                 bidirectional=True,
                 num_decoder_block=5,
                 d_decoder_ff=1024,
                     ).to(self.device)
        
        self.imagen = ElucidatedImagen(
                unets = (unet1),
                text_embed_dim = text_embed_dim,
                channels=self.pred_dim,
                channels_out=self.pred_dim,
                loss_type=self.loss_type, 
                image_sizes=[max_length],
                auto_normalize_img=False, 
                num_sample_steps=timesteps,#(64, 32), # number of sample steps - 64 for base unet, 32 for upsampler (just an example, have no clue what the optimal values are)
                sigma_min=0.002,           # min noise level
                sigma_max=80,#(80, 160),       # max noise level, @crowsonkb recommends double the max noise level for upsampler
                sigma_data=0.5,            # standard deviation of data distribution
                rho=7,                     # controls the sampling schedule
                P_mean=-1.2,               # mean of log-normal distribution from which noise is drawn for training
                P_std=1.2,                 # standard deviation of log-normal distribution from which noise is drawn for training
                S_churn=80,#40,                # parameters for stochastic sampling - depends on dataset, Table 5 in apper
                S_tmin=0.05,
                S_tmax=50,
                S_noise=1.003
        ).to (self.device)

        # self.embedding = nn.Embedding(num_embeddings=vocal_size, embedding_dim=d_emb).to (self.device)  # 假设每个时间步是一个独立的

    def forward(self, x,  output, unet_number=1): # x=contact_matrix(batch, 64, 64), output=sequece(batch, 64)(type=int) 
        # output = output.float()
        # print('PPCD output:', output.shape, output.dtype)
        # print('PPCD contact:', contact.shape, contact.dtype)
        # output = self.embedding(output)
        # print('PPCD output_embed:', output.shape, output.dtype)
        # print(x.device, edge_index.device, output.device)


        c = self.condition_process(x)
        # print(c.device)

        
 
        return self.imagen(images=output, text_embeds=c, unet_number=unet_number)
        
    # 给出条件生成一个结构
    def sample(self, x, stop_at_unet_number=1):
        # print(x.device, edge_index.device)

        c = self.condition_process(x)
        # print(c.device)
       
        return self.imagen.sample(text_embeds=c, stop_at_unet_number=stop_at_unet_number)

In [7]:
def write_fasta (sequence, filename_out):
    
    with open (filename_out, mode ='w') as f:
        f.write (f'>{filename_out}\n')
        f.write (f'{sequence}')

In [9]:
# device = torch.device('cpu')
# PPD = PPDesigner(device='cpu').to(device)
device = torch.device('cuda:1')
PPD =PPDesigner(device='cuda:1').to(device)
PPD.load_state_dict(torch.load('statedict_save-model-epoch_151.pt'))
for name, param in PPD.named_parameters():  

    param.requires_grad = False  # 冻结参数，不允许更新  
PPD.eval()

Loss type:  0
Channels in=1, channels out=1


PPDesigner(
  (condition_process): condition_process(
    (embedding): Sequential(
      (0): Linear(in_features=1, out_features=256, bias=True)
      (1): Linear(in_features=256, out_features=256, bias=True)
      (2): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
      (3): Dropout(p=0.1, inplace=False)
      (4): Sigmoid()
    )
    (PE): RotaryPositionEmbedding_learned()
  )
  (imagen): ElucidatedImagen(
    (lowres_noise_schedule): GaussianDiffusionContinuousTimes()
    (unets): ModuleList(
      (0): Unet_1d(
        (Embedding): SeqInput(
          (ffn): Sequential(
            (0): Linear(in_features=513, out_features=1024, bias=True)
            (1): Linear(in_features=1024, out_features=1024, bias=True)
            (2): SELU(inplace=True)
          )
        )
        (input_PE): RotaryPositionEmbedding_learned()
        (lstm_down): BiDirectLSTMEncoderBlockInitial(
          (lstm): LSTM(1024, 512, num_layers=5, batch_first=True, bidirectional=True)
        )
      

In [10]:
tokenizer_X_file = 'tokenizer_X.pickle'  
tokenizer_y_file = 'tokenizer_y.pickle'    
# 使用pickle加载Tokenizer模型  
with open(tokenizer_X_file, 'rb') as f:  
    tokenizer_X = pickle.load(f)  
with open(tokenizer_y_file, 'rb') as f:  
    tokenizer_y = pickle.load(f)  

In [11]:
# # test_seq = '----HHHHHHHHHHHHHHHHHHHHHHHHHHHHH----'
# test_seq = '-HHHHHHHHHHHHHHHHHHHHHH-'
# print(len(test_seq))
def seq_sample(test_seq):
    X = tokenizer_X.texts_to_sequences([test_seq])
    print(X)
    X = sequence.pad_sequences(X, maxlen=250, padding='post', truncating='post') 
    X_batch = torch.tensor(X).squeeze(-1).to(device)/10
    print(X_batch.shape)
    print(X_batch)

    output_pred = PPD.sample(
        X_batch, 
        stop_at_unet_number=1,
                                    )
                
                
    output_pred = torch.round(output_pred * 22)    
    to_rev = output_pred[0, :]
    to_rev = to_rev.long().cpu().detach().numpy().tolist()
    y_data_reversed = tokenizer_y.sequences_to_texts([to_rev])
    for index in range (len(y_data_reversed)):
        y_data_reversed[index] = y_data_reversed[index].upper().strip().replace(" ", "")
    print(y_data_reversed[0])
    
def seq_sample_truncated(test_seq):
    seq_len = len(test_seq)
    X = tokenizer_X.texts_to_sequences([test_seq])
    print(X)
    X = sequence.pad_sequences(X, maxlen=250, padding='post', truncating='post') 
    X_batch = (torch.tensor(X).squeeze(-1).to(device)/10)
    print(X_batch.shape)
    print(X_batch)
    output_pred = PPD.sample(
        X_batch, 
        stop_at_unet_number=1,
                                    )
                
                
    output_pred = torch.round(output_pred * 22)    
    to_rev = output_pred[0, :]
    to_rev = to_rev.long().cpu().detach().numpy().tolist()
    y_data_reversed = tokenizer_y.sequences_to_texts([to_rev])
    for index in range (len(y_data_reversed)):
        y_data_reversed[index] = y_data_reversed[index].upper().strip().replace(" ", "")
    print(y_data_reversed[0][0:seq_len])
    print(y_data_reversed[0])
    return y_data_reversed[0][0:seq_len]

In [12]:
# 定义要检查和创建的路径
base_path = 'SAMPLE-151'

for ind in range(16, 33, 16):
    sub_folder = 'H'+str(ind)
    file_path = os.path.join(base_path, sub_folder, sub_folder+'.txt')
     
    # 检查文件是否存在
    if not os.path.exists(file_path):
        # 文件不存在，现在检查并创建文件夹结构
        folder_path = os.path.dirname(file_path)
        
        # 使用os.makedirs创建多层文件夹结构（如果它们不存在的话）
        os.makedirs(folder_path, exist_ok=True)
        
        # 注意：这里我们并没有创建文件'H16.txt'，只是创建了文件夹结构。
        # 如果你也想创建文件，可以使用以下代码：
        # with open(file_path, 'w') as f:
        #     pass  # 这样会在指定路径创建一个空文件
     
        print(f"文件夹结构已创建：{folder_path}")
    else:
        print(f"文件已存在：{file_path}")
 
# 现在你可以继续你的逻辑，知道文件夹和（可选的）文件已经被创建或已经存在。


文件夹结构已创建：SAMPLE-151/H16
文件夹结构已创建：SAMPLE-151/H32


In [13]:
# test_seq = '----HHHHHHHHHHHHHHHHHHHHHHHHHHHHH----'
test_seq = '-' + 'H'*(16-2) +'-'
print(len(test_seq))
seq_list = []
for i in tqdm(range(50)):
    seq = seq_sample_truncated(test_seq)
    seq_list.append(seq)

with open(base_path+'/H16/H16.txt', 'w') as f:  
    for item in seq_list:  
        f.write("%s\n" % item)  

16


  0%|          | 0/50 [00:00<?, ?it/s]

[[3, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 3]]
torch.Size([1, 250])
tensor([[0.3000, 0.1000, 0.1000, 0.1000, 0.1000, 0.1000, 0.1000, 0.1000, 0.1000,
         0.1000, 0.1000, 0.1000, 0.1000, 0.1000, 0.1000, 0.3000, 0.0000, 0.0000,
         0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000, 0.0000, 0.0000, 0.00

0it [00:00, ?it/s]

sampling time step:   0%|          | 0/64 [00:00<?, ?it/s]

...

sampling time step:   0%|          | 0/64 [00:00<?, ?it/s]

SINDLDARIRSLELRI
SINDLDARIRSLELRILVLLVLLLGLSL
[[3, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 3]]
torch.Size([1, 250])
tensor([[0.3000, 0.1000, 0.1000, 0.1000, 0.1000, 0.1000, 0.1000, 0.1000, 0.1000,
         0.1000, 0.1000, 0.1000, 0.1000, 0.1000, 0.1000, 0.3000, 0.0000, 0.0000,
         0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,

0it [00:00, ?it/s]

sampling time step:   0%|          | 0/64 [00:00<?, ?it/s]

KeyboardInterrupt: 